# F8 homework: trees, ensembles and customer segments

Two parts:

- **Part A, supervised.** Compare a single decision tree, a random forest and gradient boosting on a real tabular
  dataset (breast tumour measurements, bundled with scikit-learn, so nothing to download). Then compare impurity-based
  and permutation feature importances, and catch one of them crediting pure noise.
- **Part B, unsupervised.** Segment 600 synthetic shop customers with k-means, choose $k$ with the silhouette score,
  and describe the segments in plain language.

Each task has `TODO` cells and a check cell that prints `OK` when your code is right. Run the cells in order. Everything
runs on any laptop or free Colab (CPU) in a minute or two.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

## Part A: the data (given)

569 tumours, 30 numeric measurements each, label 1 = benign, 0 = malignant. We add **two fake columns** that have
nothing to do with the label: `noise_normal` (random numbers) and `noise_id` (a random integer from 0 to 999, like a
customer ID). A good importance measure should give both roughly zero.

We hold out 25% as a test set and **do not touch it** until Task 4. All cross-validation uses the same `cv` object, so
every model sees the same folds.

In [ ]:
data = load_breast_cancer(as_frame=True)
X = data.data.copy()
y = data.target
rng = np.random.default_rng(0)
X["noise_normal"] = rng.normal(size=len(X))
X["noise_id"] = rng.integers(0, 1000, size=len(X))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
print(X_train.shape, X_test.shape)
print("majority-class accuracy on the training set:", round(max(y_train.mean(), 1 - y_train.mean()), 3))

## Task 1: a single tree, tuned by cross-validation

1. Use `GridSearchCV` with `DecisionTreeClassifier(random_state=0)`, the grid
   `{"max_depth": [2, 3, 4, 5, 6, None], "min_samples_leaf": [1, 5, 10, 20]}` and `cv=cv` (accuracy is the default
   score). Fit it on the training data and store it in `tree_search`.
2. Store the best mean CV accuracy in `tree_cv` (a float).
3. In the markdown cell below, note which settings won and whether the unlimited tree (`max_depth=None`,
   `min_samples_leaf=1`) was chosen. Why or why not (lesson 2)?

In [ ]:
tree_search = None   # TODO
tree_cv = None       # TODO

In [ ]:
assert tree_search is not None and hasattr(tree_search, "best_params_"), "Fit a GridSearchCV and store it in tree_search"
assert set(tree_search.best_params_) == {"max_depth", "min_samples_leaf"}, "Search over max_depth and min_samples_leaf"
assert tree_search.cv is cv, "Pass cv=cv so every model uses the same folds"
assert abs(tree_cv - tree_search.best_score_) < 1e-12, "tree_cv should be the best mean CV accuracy"
assert 0.85 < tree_cv < 0.99, f"A tuned tree should reach roughly 0.9 to 0.95 CV accuracy, got {tree_cv:.3f}"
print("OK", tree_search.best_params_, round(tree_cv, 3))

*Your notes on Task 1:*

## Task 2: a random forest and its out-of-bag score

1. Create `forest = RandomForestClassifier(n_estimators=300, oob_score=True, random_state=0)`.
2. Store its 5-fold CV accuracies (with `cv=cv`) in `forest_scores`, an array of 5 numbers.
3. Fit `forest` on the whole training set and compare `forest.oob_score_` with the CV mean. They estimate the same
   thing in different ways (lesson 3); are they close?

In [ ]:
forest = None          # TODO
forest_scores = None   # TODO

In [ ]:
assert forest is not None and forest.n_estimators == 300 and forest.oob_score, "Use 300 trees and oob_score=True"
assert len(forest_scores) == 5, "forest_scores should hold the 5 fold accuracies"
assert hasattr(forest, "oob_score_"), "Fit the forest on the full training set so oob_score_ exists"
assert abs(forest.oob_score_ - np.mean(forest_scores)) < 0.04, "OOB and CV accuracy should be within a few points"
assert np.mean(forest_scores) > tree_cv - 0.01, "The forest should do at least as well as the tuned tree"
print("OK", round(np.mean(forest_scores), 3), round(forest.oob_score_, 3))

## Task 3: gradient boosting

1. Create `boost = HistGradientBoostingClassifier(learning_rate=0.1, max_iter=200, random_state=0)` and store its
   CV accuracies (same `cv`) in `boost_scores`.
2. Build a DataFrame `results` with one row per model (`"tree"`, `"forest"`, `"boosting"`) as the index and columns
   `"cv_mean"` and `"cv_std"`. For the tree, use the fold scores of the best setting: they are in
   `tree_search.cv_results_["split0_test_score"]` ... `["split4_test_score"]` at index `tree_search.best_index_`.

In [ ]:
boost = None          # TODO
boost_scores = None   # TODO
tree_scores = None    # TODO: the 5 fold scores of the best tree
results = None        # TODO
results

In [ ]:
assert boost is not None and len(boost_scores) == 5, "Store the 5 CV accuracies of boosting in boost_scores"
assert np.isclose(np.mean(tree_scores), tree_cv), "tree_scores should be the fold scores of the best tree setting"
assert results is not None and set(results.index) == {"tree", "forest", "boosting"}, "results needs rows tree, forest, boosting"
assert {"cv_mean", "cv_std"} <= set(results.columns), "results needs columns cv_mean and cv_std"
assert np.isclose(results.loc["boosting", "cv_mean"], np.mean(boost_scores)), "cv_mean for boosting is wrong"
assert results.loc["tree", "cv_mean"] < max(results.loc["forest", "cv_mean"], results.loc["boosting", "cv_mean"]), "An ensemble should beat the single tree"
print("OK")

## Task 4: one look at the test set

1. Pick the model with the highest CV mean. Write its name in `best_name` (`"tree"`, `"forest"` or `"boosting"`).
2. Fit that model on the full training set (for the tree, `tree_search.best_estimator_` is already refitted) and store
   its **test** accuracy in `test_acc`.
3. In the markdown cell: is the gap between the ensembles bigger than their `cv_std`? Would you call one of them
   clearly better on this data?

In [ ]:
best_name = None   # TODO
test_acc = None    # TODO

In [ ]:
assert best_name == results["cv_mean"].idxmax(), "best_name should be the model with the highest cv_mean"
assert test_acc is not None and 0.85 < test_acc <= 1.0, "test_acc should be the chosen model's accuracy on X_test, y_test"
print("OK", best_name, round(test_acc, 3))

*Your notes on Task 4:*

## Task 5: which features matter? Impurity versus permutation

1. Put the fitted forest's `feature_importances_` into a Series `impurity` indexed by column name.
2. Compute permutation importance of the **same fitted forest on the test set** with
   `permutation_importance(forest, X_test, y_test, n_repeats=20, random_state=0)` and put `importances_mean` into a
   Series `permutation`, indexed the same way.
3. Make a horizontal bar chart of the top 10 features by impurity importance, with both measures side by side (the
   given cell does this once your Series exist).
4. Answer in the markdown cell:
   - What rank and value does `noise_id` get under each measure? Under impurity importance it outranks some real
     measurements: list a few of them with `impurity.rank(ascending=False)`. Why does impurity importance give a fake
     column credit (lesson 3: where is it computed, and what about columns with many distinct values)?
   - Many measurements here are strongly correlated (`mean radius`, `mean perimeter`, `mean area`, ...). How might that
     affect permutation importance? Compute `X_train[["mean radius", "mean perimeter", "mean area"]].corr()` to check.

In [ ]:
impurity = None      # TODO
permutation = None   # TODO

In [ ]:
assert isinstance(impurity, pd.Series) and list(impurity.index) == list(X.columns), "impurity should be a Series indexed by column name"
assert np.isclose(impurity.sum(), 1.0), "Impurity importances sum to 1"
assert isinstance(permutation, pd.Series) and list(permutation.index) == list(X.columns), "permutation should be a Series indexed by column name"
assert impurity["noise_id"] > 0.001, "Impurity importance gives the fake ID column some credit"
assert abs(permutation["noise_id"]) < 0.01 and abs(permutation["noise_normal"]) < 0.01, "Permutation importance on test data should give the noise columns about 0"
top = impurity.sort_values().tail(10).index
pd.DataFrame({"impurity": impurity[top], "permutation (test)": permutation[top]}).plot.barh(figsize=(7, 5))
plt.xlabel("importance"); plt.title("Top 10 features by impurity importance"); plt.tight_layout(); plt.show()
print("OK")

*Your answers for Task 5:*

## Part B: customer segments (given data)

A shop has four columns per customer. The generating code below plants some groups, but pretend you do not know how
many: your job is to find them.

| column | unit |
|---|---|
| `annual_spend` | euros per year |
| `visits_per_month` | visits |
| `avg_basket` | euros per visit |
| `days_since_last` | days |

In [ ]:
rng = np.random.default_rng(42)
centres = np.array([[ 600, 2.0, 25,  40],    # occasional
                    [3000, 8.0, 32,   5],    # loyal regulars
                    [2500, 1.5, 140, 30],    # big-basket
                    [ 300, 0.5, 45, 200]])   # lapsed
spread = np.array([150, 0.6, 6, 15])
sizes = [250, 150, 100, 100]
customers = pd.DataFrame(
    np.vstack([c + rng.normal(size=(n, 4)) * spread for c, n in zip(centres, sizes)]),
    columns=["annual_spend", "visits_per_month", "avg_basket", "days_since_last"]).clip(lower=0)
customers = customers.sample(frac=1, random_state=0).reset_index(drop=True)   # shuffle the rows
customers.describe().round(1)

## Task 6: scale, then choose k

1. Standardise the four columns with `StandardScaler` into an array `Z` (why is this essential here? Look at the units).
2. For each `k` in 2..8, fit `KMeans(n_clusters=k, n_init=10, random_state=0)` on `Z` and record the inertia and the
   silhouette score. Store them in lists `inertias` and `silhouettes` (index 0 is k = 2).
3. Plot both against `k` (two small plots side by side) and set `best_k` to the `k` with the highest silhouette.

In [ ]:
Z = None            # TODO
ks = list(range(2, 9))
inertias, silhouettes = [], []
# TODO: fill the two lists
best_k = None       # TODO

In [ ]:
assert Z is not None and np.allclose(np.asarray(Z).mean(axis=0), 0, atol=1e-9) and np.allclose(np.asarray(Z).std(axis=0), 1), "Z should be the standardised customer table"
assert len(inertias) == 7 and len(silhouettes) == 7, "Record one inertia and one silhouette for each k from 2 to 8"
assert all(a >= b for a, b in zip(inertias, inertias[1:])), "Inertia should never increase with k"
assert best_k == ks[int(np.argmax(silhouettes))], "best_k should be the k with the highest silhouette"
assert best_k == 4, f"On this data the silhouette should peak at k = 4, got {best_k}. Did you scale the features?"
print("OK, best k =", best_k)

## Task 7: describe the segments

1. Fit `KMeans(n_clusters=best_k, n_init=10, random_state=0)` on `Z`, and add its labels to `customers` as a column
   `"segment"`.
2. Build `profile`: the mean of each original (unscaled) column per segment, plus a `"size"` column with the number of
   customers. Describe each segment in one line in the markdown cell (for example "loyal regulars: high spend, about 8
   visits a month, seen in the last week").
3. Plot a 2-D PCA projection of `Z` coloured by segment.
4. **Contrast:** fit k-means with `best_k` on the **unscaled** columns and compute the adjusted Rand index
   (`sklearn.metrics.adjusted_rand_score`) between its labels and your scaled segments, stored in `ari_unscaled`.
   Which column dominates the unscaled distances?

In [ ]:
profile = None        # TODO
ari_unscaled = None   # TODO

In [ ]:
assert "segment" in customers and customers["segment"].nunique() == best_k, "Add a segment column with best_k segments"
assert profile is not None and "size" in profile and profile["size"].sum() == len(customers), "profile needs a size column that adds up to all customers"
assert sorted(profile["size"]) == sorted([250, 150, 100, 100]), "With scaled features k-means should recover the four planted groups exactly"
assert {"annual_spend", "visits_per_month", "avg_basket", "days_since_last"} <= set(profile.columns), "profile needs the mean of each original column"
assert ari_unscaled is not None and ari_unscaled < 0.9, "Without scaling the clustering should differ noticeably from the scaled one"
print("OK", round(ari_unscaled, 3))

*Your segment descriptions and the answer to the contrast question:*

### Stretch (optional)

1. Repeat Task 6 with an `AgglomerativeClustering(linkage="ward")` and compare the segments with k-means.
2. Drop the two noise columns and the 10 least important features (by permutation importance), and check whether the
   forest's CV accuracy changes.
3. Plot `staged_predict`-style learning curves for `GradientBoostingClassifier` with learning rates 0.3 and 0.03.